# <center> 721. Accounts Merge </center>


## Problem Description
[Click here](https://leetcode.com/problems/accounts-merge/description/)


## Intuition
<!-- Describe your first thoughts on how to solve this problem. -->
We need to merge accounts that belong to the same person. Two accounts belong to the same person if they share at least one email address.

Treat each account as a node in a graph. If two accounts share an email address, they are connected and belong to the same connected component.

We need to find the connected components in the graph. This can be done using:
- **Depth-First Search (DFS)**
    - build a graph where two accounts are connected if they share an email address
    - start DFS from an unvisited account to visit all accounts in the same connected component
    - merge all visited accounts
    - time complexity = O(V + E)
- **Union-Find (Disjoint Set)**
    - a data structure used to efficiently track groups of connected nodes in a graph
    - has two operations:
        - *find*:  finds the root (representative) of the group to determine which group a node belongs to
        - *union*: merges two different groups into one
    - to make these operations efficient, Union-Find is usually optimized using:
        - **union by size:** attach the tree with smaller size (fewer nodes) to the tree with larger size
        - **union by rank:** attach the tree with smaller rank (height) to the tree with larger rank
        - **path compression:** during the find operation, make each visited node point closer to the root, flattening the tree
        - both *union by size + path compression* and *union by rank + path compression* achieve nearly constant time per operation
    - we will use union by size and path compression


## Approach
<!-- Describe your approach to solving the problem. -->
### Union-Find Approach

#### Union-Find Class
*define a class to implement union-find. There is no need to create a separate class, we can also implement it using nested functions but it's better this way*

**init(total nodes)**
- create a list to store the parent of each node (account)
- create a list to store the size of each tree
    - *initially each node represents its own tree and acts as the root, so the size of every tree is 1*

**find(node)** <br>
*define a helper function for the find operation, it will find the root parent (root node) using path compression*
- *loop until the current node is its own parent (root node)*
    - do path compression by updating the current node's parent to its grandparent
    - update the current node
- return the root parent

**union(node1, node2)** <br>
*define a helper function for the union operation, it takes two nodes and merges the smaller size tree with the larger size*
- find the root parent of both nodes
- if both nodes have the same parent
    - return because they already belong to the same connected component
- compare the sizes of both trees
    - if the size of node1's tree is greater than or equal to the size of node2's tree
        - attach node2's tree to node1's tree by updating the parent of node2's root
        - update the size of node1's tree
    - else
        - attach node1's tree to node2's tree by updating the parent of node1's root
        - update the size of node2's tree

#### Merge Accounts Function
**accountsMerge(accounts)**
- create a Union-Find object
    - *pass the total number of accounts to create parent and size lists*
- set `email_acc` = a hashmap to map each email to the corresponding account index
    - *key = email*
    - *value = account index*
- set `acc_emails` = hashmap to map each account index to the list of corresponding emails 
    - *key = root account index*
    - *value = list of emails*
- iterate over each account and its emails to merge accounts <br>
*for each account index, (account name, emails) in the accounts list*
    - for each email
        - if the email already exists in hashmap `email_acc`
            - merge the current account and the account that already contains this email
        - else
            - add the email and current account index to the hashmap `email_acc`
- traverse `email_acc` to find the root account and its emails <br>
*for each email and account index*
    - find the root account
    - add the email to the corresponding root account in `acc_emails`
- traverse `acc_emails` to prepare the result list <br>
*for each account index and its emails*
    - sort the emails
    - create a sub-list containing the account name followed by the sorted emails
    - add it to the result list
- return the result list


## Complexity
<!-- Add your time complexity here, e.g. $$O(n)$$ -->
- Time complexity: 
    - Union-Find Approach: O(account traversal + email grouping + result creation) → O(total emails × Union-Find + unique emails × Find + sorting emails of each merged account) → O(E x α(N) + M x α(N) + MlogM) → O(E x constt + M x constt + MlogM) → O(E + M + MlogM)
        - *N = total accounts*
        - *M = total unique emails*
        - *E = total emails across all accounts*
        - *α(N) is the inverse Ackermann function, which grows extremely slowly (nearly constant)*
        - *emails are sorted within each merged account; in the worst case, all M unique emails belong to a single merged account, so the sorting cost is O(MlogM)*

<!-- Add your space complexity here, e.g. $$O(n)$$ -->
- Space complexity: 
    - Union-Find Approach: O(parent list + size list + email_acc hashmap + acc_emails hashmap + result list) → O(N + N + M + M + M) → O(N + M)
        - *parent list stores the parent of each account*
        - *size list stores the size of each tree*
        - *email_acc hashmap maps each email to an account*
        - *acc_emails hashmap groups emails belonging to the same merged account*
        - *result list stores all merged accounts and their emails*

## Code

In [ ]:
class UnionFind:

    def __init__(self, n: int):
        self.parent = [i for i in range(n)]
        self.size = [1] * n

    def find(self, x: int) -> int:
        while x != self.parent[x]:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, x1: int, x2: int) -> None:
        p1, p2 = self.find(x1), self.find(x2)
        if p1 == p2:
            return
        if self.size[p1] >= self.size[p2]:
            self.parent[p2] = p1
            self.size[p1] += self.size[p2]
        else:
            self.parent[p1] = p2
            self.size[p2] += self.size[p1]


class Solution:

    def accountsMerge(self, accounts: List[List[str]]) -> List[List[str]]:
        uf = UnionFind(len(accounts))
        email_acc = {}
        acc_emails = defaultdict(list)
        for i, (_, *emails) in enumerate(accounts):
            for e in emails:
                if e in email_acc:
                    uf.union(i, email_acc[e])
                else:
                    email_acc[e] = i
        for email, account in email_acc.items():
            acc_emails[uf.find(account)].append(email)
        return [[accounts[account][0]] + sorted(emails)
                for account, emails in acc_emails.items()]
